# POIデータから地域特徴量を作る：街を数値で表してみる

地域特徴量は「街を比較するための数値プロフィール」です。**1地域＝1行、特徴量＝列**。今回は吉祥寺駅周辺1km四方の1行を作ります。

## 1. 同じ大きさの地域で比べる準備

国土数値情報N02-25の公式駅グループ代表の駅線中点を中心に、EPSG:6677上で東西・南北それぞれ500m。対象面積は1.0km²です。駅の出入口や商圏を表す範囲ではありません。取得済みOverture Placesのreleaseは2026-09-23.0に固定し、このNotebookから取得はしません。

250m格子16区画・500m格子4区画は算出根拠となる途中データです。部分区画はありません。

[準備・出典・再現手順](../docs/region-features.md) / [列の意味](../docs/region-feature-columns.md) / [検証](../docs/region-features-validation.md)

In [ ]:
from geoai_open_lab.region_features import run_experiment
from geoai_open_lab.region_figures import save_figures
import pandas as pd
from IPython.display import SVG, display

results, manifest, output_dir = run_experiment()
figures = save_figures(results, output_dir)

## 2. 吉祥寺駅周辺を1行の数値にする

密度は登録数÷面積、構成比は各群の件数÷全登録数（欠損を含む）。集中度は250m格子の最大密度÷地域全体の平均密度です。

比較候補は密度・カテゴリ構成比・集中度。面積・登録数・欠損率、manifestのrelease・取得日時・AOIは監査にも使います。登録数と密度を重複投入せず、実際の採用列や正規化は複数地域で分布を確認してから決めます。

In [ ]:
with pd.option_context("display.max_columns", None):
    display(pd.read_csv(output_dir / "region_features.csv"))

## 3. 数値プロフィールを読む

表の列を日本語・単位付きで示します。食事群などの定義は設定のdisplay_groupsが正本。「その他」は未掲載の既知カテゴリで、欠損とは別です。

In [ ]:
display(SVG(filename=str(figures[2])))

## 4. 250m格子で算出根拠を確認する

北が上、赤点が吉祥寺駅の代表点。色は登録密度、数字は登録件数です。駅中心の1km四方を同じ大きさの区画に分けています。500m集計でも全体の登録数は一致します。格子原点を駅に揃えても、最大密度は区切り方に敏感です。

In [ ]:
display(SVG(filename=str(figures[0])))

## 5. 欠損・営業状態・範囲の限界

営業状態が欠損している登録を営業中とは判定しません。POI登録数は実際の全店舗数ではありません。位置やカテゴリの精度、実店舗の重複は未検証です。図はEPSG:6677の北を上にした分析用の格子で、背景地図は使いません。

In [ ]:
print("営業状態の内訳:", manifest["status_counts"])
print("両解像度の登録数:", {s: int(c.poi_count.sum()) for s, (c, _) in results.items()})

## 6. 次は東京都内の駅を同じ条件で比較する

23区だけでなく多摩地域も対象に、駅中心1km四方を同じ列で集計する予定です。重なるAOIへの同一POIの登録を許容します。「その他」の分布も確認したうえで、特徴量選択・正規化・距離関数を決めます。今回は東京都全体の取得・類似度計算は行っていません。

[将来計画](../docs/tokyo-station-features-plan.md) / [記事引き継ぎ](../docs/region-features-article-handoff.md)。公開Notebookは出力なし。実行済みNotebook・HTML・CSV・SVGはGit管理外のdataへ保存します。